# 07 · Binary classification

Same training loop as regression — three things change: the **output layer**, the **loss**, and the **way you judge the result** (a loss of 0.45 tells a human nothing; accuracy, precision and recall do).

Dataset: the Titanic passenger list used in the lecturer's week 6–8 notebooks. Label: `Survived` (0/1).

---
## Theory in 60 seconds

- Output layer: **one neuron**. Its raw output `z` is called the **logit**; `p = σ(z) = 1/(1+e^{−z})` is the predicted probability of class 1.
- Loss: **binary cross-entropy** `−[y·ln p + (1−y)·ln(1−p)]`, averaged over the batch. It is the negative log of the probability the model assigned to the *correct* answer.
- Two equivalent ways to wire it in PyTorch — **never mix them**:

| | last layer of the model | loss | model output is |
|---|---|---|---|
| lecturer's notebooks | `nn.Linear(…, 1)` + `nn.Sigmoid()` | `nn.BCELoss()` | a probability |
| lab solutions | `nn.Linear(…, 1)` | `nn.BCEWithLogitsLoss()` | a logit |

- Labels for both: `float32`, same shape as the output, `(batch, 1)`.
- Decision: class 1 if `p > 0.5` ⇔ `z > 0`.
- **Accuracy** = correct / all. With imbalanced classes it flatters a useless model, so also look at **precision** = TP/(TP+FP) ("when I say 1, how often am I right?") and **recall** = TP/(TP+FN) ("of the real 1s, how many did I find?"), and **F1** = their harmonic mean.

In [ ]:
import sys; sys.path.insert(0, "..")
import csv
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from dnd_utils import check, DATA, load_titanic_raw, load_titanic

with open(DATA / "titanic.csv") as f:
    for _ in range(4): print(f.readline().rstrip())

---
## Exercise 7.1 · Parsing a CSV with quoted fields

The `Name` field contains commas inside quotes, so `line.split(",")` breaks the columns. Python's built-in `csv` module handles this.

Build, with `csv.reader`:

- `inputs`: `(891, 7)` `float32` with the columns **Pclass, Sex, Age, SibSp, Parch, Fare, Embarked** — `Sex`: female → 0, male → 1; `Embarked`: C → 0, Q → 1, S → 2; empty fields → `np.nan`
- `labels`: `(891,)` `int32`, the `Survived` column

<details><summary>Hint</summary>

```python
with open(path, newline="") as f:
    reader = csv.reader(f)
    header = next(reader)          # first row
    for row in reader:             # each row is a list of strings, quotes already handled
        ...
```
Find the column positions with `header.index("Age")` instead of counting by hand.
</details>

In [ ]:
inputs = None
labels = None

In [ ]:
_ri, _rl = load_titanic_raw()
check("inputs: shape and dtype", lambda: inputs.shape == (891, 7) and inputs.dtype == np.float32)
check("inputs: values", lambda: np.allclose(inputs, _ri, equal_nan=True))
check("labels", lambda: labels.shape == (891,) and labels.dtype == np.int32 and (labels == _rl).all())

The rest of the preprocessing is what you built in notebook 04 (flag + median for `Age`, one-hot for `Embarked`, shuffle, 60/20/20 split, standardisation with training statistics). To keep this notebook focused it is loaded ready-made:

In [ ]:
d = load_titanic()
X_train, y_train = d["X_train"], d["y_train"]        # (534, 10) float32, (534,) float32 in {0, 1}
X_val, y_val = d["X_val"], d["y_val"]
X_test, y_test = d["X_test"], d["y_test"]
print(X_train.shape, y_train.shape, y_train.dtype, "| columns: Pclass Age SibSp Parch Fare | Sex | Emb C Q S | Age-missing")

---
## Exercise 7.2 · Know your classes before you train

Compute the fraction of survivors in the training set, and `baseline_acc`: the **test** accuracy of the laziest classifier — always predict the class that is the majority in the **training** set.

In [ ]:
pos_rate_train = None
baseline_acc = None

In [ ]:
check("pos_rate_train ≈ 0.36", lambda: abs(pos_rate_train - y_train.mean()) < 1e-6)
check("baseline_acc", lambda: abs(baseline_acc - (y_test == 0).mean()) < 1e-6)

Any model you train must be judged against this number, not against 50 %.

---
## Exercise 7.3 · Binary cross-entropy by hand

**a)** Implement `my_bce(p, y)`: mean BCE from **probabilities** (tensors of equal shape).

**b)** Implement `my_bce_with_logits(z, y)`: mean BCE from **logits**, in the numerically stable form

$$\ell(z, y) = \max(z, 0) - z\,y + \ln\!\big(1 + e^{-|z|}\big)$$

(Optional, on paper: show that this equals `−[y ln σ(z) + (1−y) ln(1−σ(z))]`.)

In [ ]:
def my_bce(p, y):
    raise NotImplementedError

def my_bce_with_logits(z, y):
    raise NotImplementedError

In [ ]:
torch.manual_seed(0)
_z = torch.randn(50, 1) * 3
_y = (torch.rand(50, 1) < 0.4).float()
check("my_bce == nn.BCELoss", lambda: torch.allclose(my_bce(torch.sigmoid(_z), _y), nn.BCELoss()(torch.sigmoid(_z), _y), atol=1e-5))
check("my_bce_with_logits == nn.BCEWithLogitsLoss", lambda: torch.allclose(my_bce_with_logits(_z, _y), nn.BCEWithLogitsLoss()(_z, _y), atol=1e-6))
check("both routes give the same number", lambda: torch.allclose(my_bce(torch.sigmoid(_z), _y), my_bce_with_logits(_z, _y), atol=1e-5))

**c)** Fill in the table by reasoning (then verify with your function): the loss of a **single** sample with true label `y = 1` when the model predicts the given probability.

In [ ]:
loss_when_p_is = {0.99: None, 0.5: None, 0.01: None}       # rounded to 2 decimals

In [ ]:
for p_, l_ in loss_when_p_is.items():
    check(f"p = {p_}: loss = {-np.log(p_):.2f}", lambda: abs(l_ - (-np.log(p_))) < 0.006)

The loss is **unbounded** for confident wrong answers: one sample predicted with `p = 0.01` costs as much as 460 samples predicted with `p = 0.99`. Cross-entropy punishes over-confidence — that is its job.

**d)** Why does the logits version exist at all? Run the cell and explain the difference in the last column.

In [ ]:
for z_val in (-5.0, -20.0, -40.0):
    z1 = torch.tensor([[z_val]], requires_grad=True)
    nn.BCELoss()(torch.sigmoid(z1), torch.ones(1, 1)).backward()
    z2 = torch.tensor([[z_val]], requires_grad=True)
    nn.BCEWithLogitsLoss()(z2, torch.ones(1, 1)).backward()
    print(f"logit {z_val:6.1f} (true label 1)   gradient via Sigmoid+BCELoss: {z1.grad.item():10.6f}   via BCEWithLogitsLoss: {z2.grad.item():10.6f}")

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 7.4 · Two models, two accuracy functions

Architecture `10 → 30 → 20 → 1` with ReLU.

- `BinMLPProb`: ends with a sigmoid (outputs probabilities) — to be used with `nn.BCELoss`.
- `BinMLPLogit`: no final activation (outputs logits) — to be used with `nn.BCEWithLogitsLoss`.
- `acc_from_probs(p, y)` and `acc_from_logits(z, y)`: fraction of correct predictions as a tensor scalar or float; inputs have shape `(batch, 1)`.

In [ ]:
class BinMLPProb(nn.Module):
    pass

class BinMLPLogit(nn.Module):
    pass

def acc_from_probs(p, y):
    raise NotImplementedError

def acc_from_logits(z, y):
    raise NotImplementedError

In [ ]:
_x = torch.randn(64, 10)
check("BinMLPProb outputs probabilities, shape (64, 1)", lambda: (lambda o: o.shape == (64, 1) and 0 < o.min() and o.max() < 1)(BinMLPProb(10)(_x)))
check("BinMLPLogit outputs unbounded logits, shape (64, 1)", lambda: (lambda o: o.shape == (64, 1) and o.min() < 0)((torch.manual_seed(1), BinMLPLogit(10))[1](_x * 5)))
_p = torch.tensor([[0.9], [0.2], [0.6], [0.4]]); _yy = torch.tensor([[1.0], [0.0], [0.0], [0.0]])
check("acc_from_probs", lambda: abs(float(acc_from_probs(_p, _yy)) - 0.75) < 1e-6)
check("acc_from_logits", lambda: abs(float(acc_from_logits(torch.logit(_p), _yy)) - 0.75) < 1e-6)

---
## Exercise 7.5 · Train it

1. Write the `Dataset` (labels `float32`, batches `(batch, 1)`) and three loaders (batch size 32).
2. Train a fresh `BinMLPLogit` for 60 epochs with `BCEWithLogitsLoss` and `Adam(lr=0.001)`.
3. Per epoch record `train_losses`, `val_losses`, `train_accs`, `val_accs` (mean over batches is fine).
4. **Before** the first update, store the validation loss of the untrained model in `initial_val_loss`.

In [ ]:
torch.manual_seed(0)
model = None
initial_val_loss = None
train_losses, val_losses, train_accs, val_accs = [], [], [], []

In [ ]:
check("initial loss is close to ln 2 = 0.693", lambda: abs(initial_val_loss - np.log(2)) < 0.08)
check("60 epochs recorded", lambda: len(train_losses) == len(val_losses) == len(train_accs) == len(val_accs) == 60)
check("training loss went down clearly", lambda: train_losses[-1] < 0.5)
check("validation accuracy clearly above the baseline", lambda: val_accs[-1] > 0.75)

Plot two panels side by side: losses (train/val) and accuracies (train/val), each with a legend.

<details><summary>Hint</summary>

`fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))`, then `ax1.plot(...)`, `ax1.set_xlabel(...)`, `ax1.legend()`.
</details>

In [ ]:
# your plot

The loss curve is smooth, the accuracy curve is jagged, and sometimes the loss improves while the accuracy does not move. Why — and why do we train on the loss rather than directly on accuracy?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 7.6 · Evaluate on the test set

Compute on the full test set (concatenate the batches):

- `test_logits`, `test_true`: tensors `(179, 1)`
- `test_loss`, `test_acc`
- the four confusion counts `TP, FP, FN, TN` (Python ints; "positive" = survived = 1)
- `precision`, `recall`, `f1`

In [ ]:
test_logits = test_true = None
test_loss = test_acc = None
TP = FP = FN = TN = None
precision = recall = f1 = None

In [ ]:
check("shapes (179, 1)", lambda: test_logits.shape == test_true.shape == (179, 1))
check("the four counts add up to 179", lambda: TP + FP + FN + TN == 179)
check("accuracy == (TP + TN) / n", lambda: abs(test_acc - (TP + TN) / 179) < 1e-6)
check("TP + FN == number of survivors in the test set", lambda: TP + FN == int(y_test.sum()))
check("precision, recall, f1 consistent", lambda: abs(precision - TP / (TP + FP)) < 1e-9 and abs(recall - TP / (TP + FN)) < 1e-9 and abs(f1 - 2 * TP / (2 * TP + FP + FN)) < 1e-9)
check("clearly better than the majority baseline", lambda: test_acc > baseline_acc + 0.1)

Your model probably has **higher precision than recall**: it misses a fair number of survivors. One reason is the class imbalance (62 % / 38 %) — when unsure, guessing "did not survive" is the safer bet for the loss. The next exercises are the two standard remedies from the lecture.

---
## Exercise 7.7 · Remedy 1: a weighted loss

Idea: multiply the loss of every **positive** sample by `w_pos > 1`, so that mistakes on the minority class cost more.

**a)** Implement `weighted_bce_with_logits(z, y, w_pos)` using `nn.BCEWithLogitsLoss(reduction="none")`: get the per-sample losses, multiply by a weight tensor that is `w_pos` where `y == 1` and `1` elsewhere, then take the mean.

In [ ]:
def weighted_bce_with_logits(z, y, w_pos):
    raise NotImplementedError

In [ ]:
check("w_pos = 1 gives the ordinary loss", lambda: torch.allclose(weighted_bce_with_logits(_z, _y, 1.0), nn.BCEWithLogitsLoss()(_z, _y)))
check("equals PyTorch's built-in pos_weight", lambda: torch.allclose(weighted_bce_with_logits(_z, _y, 3.0), nn.BCEWithLogitsLoss(pos_weight=torch.tensor(3.0))(_z, _y), atol=1e-6))
check("returns a scalar", lambda: weighted_bce_with_logits(_z, _y, 3.0).ndim == 0)

**b)** The shape trap of this technique. `per_sample` has shape `(batch, 1)`. Suppose your weights came from a 1-D label array and have shape `(batch,)`. Predict the shape of `per_sample * weights` and whether the resulting "loss" is still correct.

In [ ]:
trap_shape_for_batch_50 = None       # a tuple
trap_is_correct = None               # True / False

In [ ]:
_per = nn.BCEWithLogitsLoss(reduction="none")(_z, _y); _w1d = torch.where(_y[:, 0] == 1, 3.0, 1.0)
check("shape", lambda: tuple(trap_shape_for_batch_50) == tuple((_per * _w1d).shape))
check("correctness", lambda: trap_is_correct is False and not torch.allclose((_per * _w1d).mean(), weighted_bce_with_logits(_z, _y, 3.0)))
print("buggy value:", (_per * _w1d).mean().item(), "= mean(loss) * mean(weight) =", (_per.mean() * _w1d.mean()).item(), "| correct:", weighted_bce_with_logits(_z, _y, 3.0).item())

**c)** Train a fresh `BinMLPLogit` exactly as in 7.5, but with the weighted loss and `w_pos = (number of negatives) / (number of positives)` in the training set. Then compute `recall_w`, `precision_w`, `acc_w` on the test set.

<details><summary>Hint</summary>

Your training code takes a `loss_fn(z, y)`. Either pass `nn.BCEWithLogitsLoss(pos_weight=torch.tensor(w_pos))`, or wrap your own function: `lambda z, y: weighted_bce_with_logits(z, y, w_pos)`.
</details>

In [ ]:
w_pos = None
recall_w = precision_w = acc_w = None

In [ ]:
check("w_pos = n_neg / n_pos", lambda: abs(w_pos - (y_train == 0).sum() / (y_train == 1).sum()) < 1e-6)
check("recall did not get worse (usually it rises)", lambda: recall_w >= recall - 0.02)
check("still a sensible classifier", lambda: acc_w > baseline_acc + 0.08)

Weighting typically **raises recall and lowers precision**: the model says "survived" more readily. Whether that is an improvement depends on which mistake is more expensive in the application — it is a decision, not a free lunch.

---
## Exercise 7.8 · Remedy 2: balanced mini-batches

Instead of re-weighting the loss, feed the network batches with **equally many samples of each class**.

Write the generator `balanced_batches(X, y, batch_size, n_batches, rng)`:

- `X`: tensor `(n, d)`; `y`: tensor `(n,)` with values 0.0/1.0; `batch_size` is even
- every batch contains `batch_size // 2` randomly chosen positives and `batch_size // 2` randomly chosen negatives (sampling **with** replacement, so the minority class may repeat)
- yields `(xb, yb)` with `yb` of shape `(batch_size, 1)`

<details><summary>Hint</summary>

Compute the index arrays of each class once: `pos_idx = np.where(y.numpy() == 1)[0]`. In each batch draw `rng.choice(pos_idx, size=batch_size // 2)` and the same for the negatives, concatenate the two index arrays, index `X` and `y`.
</details>

In [ ]:
def balanced_batches(X, y, batch_size, n_batches, rng):
    raise NotImplementedError
    yield

In [ ]:
_Xt, _yt = torch.tensor(X_train), torch.tensor(y_train)
_bb = None
try: _bb = list(balanced_batches(_Xt, _yt, 32, 10, np.random.default_rng(0)))
except NotImplementedError: pass
check("10 batches of shapes (32, 10) and (32, 1)", lambda: len(_bb) == 10 and all(x.shape == (32, 10) and y.shape == (32, 1) for x, y in _bb))
check("every batch is exactly half positive", lambda: all(y.sum().item() == 16 for _, y in _bb))
check("inputs and labels stay paired", lambda: all(any(torch.equal(_bb[0][0][i], _Xt[j]) for j in (_yt == _bb[0][1][i, 0]).nonzero()[:, 0]) for i in (0, 20)))
check("batches differ from each other", lambda: not torch.equal(_bb[0][0], _bb[1][0]))

With such a generator there is no natural "epoch" any more (samples are drawn at random with replacement). How would you decide how many batches make up one epoch? And should the **validation** data also be balanced this way?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 7.9 · Moving the threshold

The 0.5 threshold is a choice. Write `prf_at(logits, y, threshold)` that returns `(precision, recall)` when class 1 is predicted for `p > threshold`. If the model predicts no positives at all, return precision `0.0`. Use it on the **plain** model's test logits for thresholds 0.3, 0.5 and 0.7.

In [ ]:
def prf_at(logits, y, threshold):
    raise NotImplementedError

In [ ]:
for thr in (0.3, 0.5, 0.7):
    try: print(f"threshold {thr}: precision %.3f  recall %.3f" % prf_at(test_logits, test_true, thr))
    except Exception as e: print(type(e).__name__)
check("threshold 0.5 reproduces 7.6", lambda: np.allclose(prf_at(test_logits, test_true, 0.5), (precision, recall)))
check("lower threshold -> recall does not decrease", lambda: prf_at(test_logits, test_true, 0.3)[1] >= prf_at(test_logits, test_true, 0.5)[1] >= prf_at(test_logits, test_true, 0.7)[1])
check("no positives predicted -> precision 0.0, no crash", lambda: prf_at(test_logits, test_true, 1.0) == (0.0, 0.0))

---
## Exercise 7.10 · Bug hunt: the double sigmoid

Someone combined the two recipes: a model that ends in `nn.Sigmoid()` **and** `nn.BCEWithLogitsLoss`. There is no error and no warning.

**a)** The loss function now receives a number between 0 and 1 and treats it as a logit. What is the smallest loss it can ever give for a sample with `y = 1`? For a sample with `y = 0`? (Use the formula `ℓ = ln(1 + e^{−z})` for `y = 1` and `ℓ = ln(1 + e^{z})` for `y = 0`, with `z` restricted to `[0, 1]`.)

In [ ]:
min_loss_y1 = None
min_loss_y0 = None

In [ ]:
check("min loss for y = 1 is 0.313", lambda: abs(min_loss_y1 - 0.3133) < 1e-3)
check("min loss for y = 0 is 0.693", lambda: abs(min_loss_y0 - 0.6931) < 1e-3)

**b)** Run the buggy training below. Then explain the two fingerprints: the value at which the loss gets stuck, and the accuracy you get if you (consistently with the loss) threshold `sigmoid(output) > 0.5`.

In [ ]:
torch.manual_seed(0)
buggy = BinMLPProb(X_train.shape[1])                       # ends with Sigmoid ...
buggy_loss = nn.BCEWithLogitsLoss()                        # ... and the loss applies a second one
buggy_opt = torch.optim.Adam(buggy.parameters(), lr=0.001)
try:
    for epoch in range(60):
        buggy.train()
        ep = []
        for xb, yb in loader_train:
            loss = buggy_loss(buggy(xb), yb)
            buggy_opt.zero_grad(); loss.backward(); buggy_opt.step()
            ep.append(loss.item())
        if epoch % 20 == 0 or epoch == 59: print(f"epoch {epoch:2d}  training loss {np.mean(ep):.3f}")
    with torch.no_grad():
        out = buggy(torch.tensor(X_test))
    print("accuracy with sigmoid(output) > 0.5:", ((torch.sigmoid(out) > 0.5).float()[:, 0] == torch.tensor(y_test)).float().mean().item())
    print("accuracy with        output  > 0.5:", ((out > 0.5).float()[:, 0] == torch.tensor(y_test)).float().mean().item())
    print("theoretical floor of the loss: %.3f" % (y_train.mean() * 0.3133 + (1 - y_train.mean()) * 0.6931))
except NameError as e:
    print("finish the exercises above first:", e)

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Blank-page drill

New empty notebook, 25 minutes, nothing to look at. Starting from `d = load_titanic()`:

1. dataset + loaders;
2. a model `10 → 30 → 20 → 1` — this time in the **lecturer's style**: `nn.Sigmoid()` at the end and `nn.BCELoss()`;
3. a training loop recording loss **and accuracy** for training and validation;
4. the two-panel plot;
5. test loss, test accuracy, and precision / recall / F1 computed by hand;
6. one sentence: is the model better than the majority baseline?

The solution notebook contains this variant at the end.

In [ ]:
# (do this one in a new notebook)